In [1]:
from ase import Atoms
from ase.io import write
from ase.visualize import view as ase_view
import numpy as np
import nglview as nv

# Parameters
a = 2.5  # hBN lattice constant (Å)
vacuum = 15.0  # Vacuum in z-direction (Å)
repeats = (6, 6)  # Supercell dimensions
wt_percent_target = 1.0  # Target CuO weight percent

# Step 1: Build unit cell of hBN
positions = [(0, 0, 0), (1/3, 2/3, 0)]
symbols = ['B', 'N']
cell = [[a, 0, 0],
        [-a/2, a * np.sqrt(3)/2, 0],
        [0, 0, 10]]

hbn = Atoms(symbols=symbols,
            scaled_positions=positions,
            cell=cell,
            pbc=[1, 1, 0])
hbn *= (*repeats, 1)
hbn.center(vacuum=vacuum, axis=2)

# Step 2: Create CuO molecule manually
cuo = Atoms('CuO', positions=[[0, 0, 0], [0, 0, 1.73]])

# Step 3: Calculate number of CuO molecules to match ~1 wt%
mass_hbn = hbn.get_masses().sum()
mass_cuo = cuo.get_masses().sum()
target_mass_cuo = wt_percent_target / 100 * mass_hbn
num_cuo = int(round(target_mass_cuo / mass_cuo))

print(f"Mass of hBN: {mass_hbn:.2f} u")
print(f"Mass of CuO: {mass_cuo:.2f} u")
print(f"Target CuO mass: {target_mass_cuo:.2f} u → {num_cuo} molecule(s)")

# Step 4: Place CuO molecules above the hBN surface
cell_x, cell_y, cell_z = hbn.get_cell().diagonal()
z_offset = cell_z / 2 + 2.5  # 2.5 Å above top hBN layer

# Rotation matrix: 90° about x-axis
rotation_matrix = np.array([[1, 0, 0],
                            [0, 0, -1],
                            [0, 1, 0]])

for i in range(num_cuo):
    cuo_i = cuo.copy()
    new_positions = cuo_i.get_positions().dot(rotation_matrix.T)
    cuo_i.set_positions(new_positions)

    x = (i + 1) * cell_x / (num_cuo + 1)
    y = cell_y / 2
    cuo_i.translate([x, y, z_offset])
    hbn += cuo_i

# Step 5: Save structure
write('cuo_on_hbn.xyz', hbn)
write('cuo_on_hbn.vasp', hbn, direct=True)

# Step 6: Visualize with nglview (in Jupyter Notebook)
nv_view = nv.show_ase(hbn)
nv_view.representations = [{"type": "ball+stick", "params": {"sele": "all"}}]
nv_view


ModuleNotFoundError: No module named 'ase'

In [ ]:
from ase import Atoms
from ase.io import write
import numpy as np
import nglview as nv
import random

# Parameters
a = 2.5  # hBN lattice constant (Å)
vacuum = 15.0  # Vacuum in z-direction (Å)
target_b_n_mass = 48000  # Target mass of Boron and Nitrogen in hBN (u)
wt_percent_target = 1.0  # Target CuO weight percent

# Step 1: Build unit cell of hBN
positions = [(0, 0, 0), (1/3, 2/3, 0)]
symbols = ['B', 'N']
cell = [[a, 0, 0],
        [-a/2, a * np.sqrt(3)/2, 0],
        [0, 0, 10]]

# Calculate the mass of one hBN unit cell (B + N atoms)
mass_b_n_per_unit = 10.81 + 14.01  # Mass of B and N in a single unit cell

# Calculate how many unit cells are needed for 48000 u of B and N
num_hbn_units = int(np.ceil(target_b_n_mass / mass_b_n_per_unit))

# Step 2: Scale the hBN supercell to have the correct number of units
# Choose square root of the required number of units for repeats to balance the supercell
repeats = (int(np.sqrt(num_hbn_units)), int(np.sqrt(num_hbn_units)))  # Adjust repeats to achieve target mass

hbn = Atoms(symbols=symbols,
            scaled_positions=positions,
            cell=cell,
            pbc=[1, 1, 0])
hbn *= (*repeats, 1)
hbn.center(vacuum=vacuum, axis=2)

# Step 3: Create CuO molecule manually
cuo = Atoms('CuO', positions=[[0, 0, 0], [0, 0, 1.73]])

# Step 4: Calculate number of CuO molecules to match ~1 wt%
mass_hbn = hbn.get_masses().sum()
mass_cuo = cuo.get_masses().sum()
target_mass_cuo = wt_percent_target / 100 * mass_hbn
num_cuo = int(round(target_mass_cuo / mass_cuo))

print(f"Mass of hBN: {mass_hbn:.2f} u")
print(f"Mass of CuO: {mass_cuo:.2f} u")
print(f"Target CuO mass: {target_mass_cuo:.2f} u → {num_cuo} molecule(s)")

# Step 5: Place CuO molecules randomly on the surface of hBN at a fixed z_offset (3 Å)
cell_x, cell_y, cell_z = hbn.get_cell().diagonal()

# Set fixed z_offset to 3 Å above the surface
z_offset = 3.0  # Fixed distance above the surface

# Get the positions of all B and N atoms in the supercell
b_positions = hbn.get_positions()[::2]  # B atoms are at even indices
n_positions = hbn.get_positions()[1::2]  # N atoms are at odd indices

# Combine positions of B and N atoms
all_positions = np.vstack((b_positions, n_positions))

# Loop through and randomly shuffle the positions to distribute CuO molecules across the surface
random.shuffle(all_positions)

# Rotation matrix (optional for different orientations of CuO)
rotation_matrix = np.array([[1, 0, 0],
                            [0, 0, -1],
                            [0, 1, 0]])

# Now randomly place CuO molecules across the entire surface
positions_placed = 0
for pos in all_positions:
    if positions_placed < num_cuo:
        cuo_i = cuo.copy()
        new_positions = cuo_i.get_positions().dot(rotation_matrix.T)
        cuo_i.set_positions(new_positions)

        # Randomly place CuO molecules within the x and y bounds of the supercell
        x_random = random.uniform(0, cell_x)
        y_random = random.uniform(0, cell_y)

        # Place CuO molecule directly above the surface at the z_offset distance
        cuo_i.translate([x_random, y_random, z_offset])
        hbn += cuo_i
        positions_placed += 1

# Step 6: Save structure
write('cuo_on_hbn_random_above_B_and_N_surface_fully_randomized.xyz', hbn)
write('cuo_on_hbn_random_above_B_and_N_surface_fully_randomized.vasp', hbn, direct=True)

# Step 7: Visualize with nglview (in Jupyter Notebook)
nv_view = nv.show_ase(hbn)
nv_view.representations = [{"type": "ball+stick", "params": {"sele": "all"}}]
nv_view


Mass of hBN: 45886.63 u
Mass of CuO: 79.55 u
Target CuO mass: 458.87 u → 6 molecule(s)


NGLWidget()

In [ ]:
from ase import Atoms
from ase.build import nanotube
from ase.io import write
import numpy as np
import nglview as nv

# Parameters
n, m = 10, 10
length = 90  # 3x longer length
wt_percent_target = 1.0
cuo_spacing_r = 2.0

# Step 1: Build carbon nanotube first
cnt = nanotube(n, m, length=length, symbol='C')

# Step 2: Replace atoms alternately with B and N
symbols = cnt.get_chemical_symbols()
new_symbols = ['B' if i % 2 == 0 else 'N' for i in range(len(symbols))]
cnt.set_chemical_symbols(new_symbols)

cnt.center(vacuum=20.0, axis=(0, 1, 2))

# Step 3: CuO molecule upright inside tube
cuo = Atoms('CuO', positions=[[0, 0, 0], [0, 0, 1.73]])
rotation_matrix = np.array([[1, 0, 0],
                            [0, 0, -1],
                            [0, 1, 0]])
cuo.set_positions(cuo.get_positions().dot(rotation_matrix.T))

# Step 4: Calculate number of CuO molecules
mass_bnnt = cnt.get_masses().sum()
mass_cuo = cuo.get_masses().sum()
target_mass_cuo = wt_percent_target / 100 * mass_bnnt
num_cuo = int(round(target_mass_cuo / mass_cuo))

# Step 5: Generate random internal positions for CuO
np.random.seed(42)
positions = cnt.get_positions()
center = positions.mean(axis=0)
r_inner = np.linalg.norm(positions[0][:2] - center[:2]) - 1.0

ads_sites = []
for _ in range(num_cuo):
    angle = np.random.uniform(0, 2 * np.pi)
    z = np.random.uniform(center[2] - length / 2 + 5, center[2] + length / 2 - 5)
    x = center[0] + (r_inner - cuo_spacing_r) * np.cos(angle)
    y = center[1] + (r_inner - cuo_spacing_r) * np.sin(angle)
    ads_sites.append((x, y, z))

# Step 6: Place CuO molecules inside BNNT
for site in ads_sites:
    cuo_i = cuo.copy()
    cuo_i.translate(site)
    cnt += cuo_i

# Step 7: Save structure and visualize
write('cuo_inside_bnnt.xyz', cnt)
write('cuo_inside_bnnt.vasp', cnt, direct=True)

nv_view = nv.show_ase(cnt)
nv_view.clear_representations()
nv_view.add_ball_and_stick()
nv_view.center()
nv_view


NGLWidget()

In [ ]:
from pymatgen import MPRester
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, f1_score
import matplotlib.pyplot as plt

# Materials Project API key (replace with your own)
api_key = "your_api_key"  # Insert your API key here
mpr = MPRester(api_key)

# Define the materials we're interested in (CuO, hBN, BNNT)
materials = ['CuO', 'hBN', 'BNNT']

# Get the material data from the Materials Project database
data = []
for material in materials:
    # Query the database for each material
    entries = mpr.query({"material_ids": material}, properties=["structure", "energy", "band_gap", "formation_energy_per_atom", "density"])
    
    # Store the queried data
    for entry in entries:
        data.append({
            "material": material,
            "energy": entry["energy"],  # Total energy
            "band_gap": entry["band_gap"],  # Band gap
            "formation_energy": entry["formation_energy_per_atom"],  # Formation energy
            "density": entry["density"]  # Density
        })

# Convert data to a DataFrame for easier processing
df = pd.DataFrame(data)

# Check for missing values and handle them
df = df.dropna()  # Drop rows with missing values

# Feature extraction: Using energy, band_gap, and formation_energy as features
features = df[["energy", "band_gap", "formation_energy", "density"]]  # Features for training
target = df["energy"]  # Energy is our target variable

# Split data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the Random Forest Regressor model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model using Mean Squared Error and R2 score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Output evaluation metrics
print(f"Mean Squared Error: {mse}")
print(f"R2 Score: {r2}")

# F1 score (requires converting continuous output to a classification problem)
# For demonstration purposes, let's convert energy values to categories for F1 scoring
# Here, we assume energies below a certain threshold are "low energy", and above are "high energy"
threshold = np.median(y_test)
y_test_classified = (y_test > threshold).astype(int)  # Classify as 1 (high energy) or 0 (low energy)
y_pred_classified = (y_pred > threshold).astype(int)

f1 = f1_score(y_test_classified, y_pred_classified)
print(f"F1 Score: {f1}")

# Plot actual vs predicted values
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Energy (eV)")
plt.ylabel("Predicted Energy (eV)")
plt.title("Actual vs Predicted Energy")
plt.show()

# Example: Predict energy for a new material (hypothetical data)
new_material_data = np.array([[0.6, 3.2, -2.0, 2.0]])  # Example: [band_gap, formation_energy, energy, density]
new_material_scaled = scaler.transform(new_material_data)

# Predict energy for the new material
predicted_energy = model.predict(new_material_scaled)
print(f"Predicted Energy for New Material: {predicted_energy[0]} eV")


ImportError: cannot import name 'MPRester' from 'pymatgen' (unknown location)